In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
import json
import dask
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from xcube.core.store import new_data_store
from xcube.core.chunk import chunk_dataset
from xcube.core.gridmapping import GridMapping
from xcube.core.geom import mask_dataset_by_geometry
from xcube_resampling.spatial import resample_in_space
from xcube_resampling.gridmapping import GridMapping
from dask.distributed import Client, LocalCluster

In [ ]:
INPUT_DIR = "input_irrigation_10years"

In [ ]:
irr_store = new_data_store("file", root=INPUT_DIR)

In [ ]:
# bbox = [-5, 40, 3, 44] # Ebro Basin
# time_range = ("2020-01-01", "2021-12-31")
# time_range = ("2020-01-01", "2020-01-31")
bbox = [-31, 27, 40, 81] # Europe
time_range = ("2016-01-01", "2025-09-30")

In [ ]:
json_file_path = "credentials.json"
with open(json_file_path, "r") as j:
    credentials = json.loads(j.read())

In [ ]:
import logging
logging.getLogger().setLevel(logging.WARNING)
LOG = logging.getLogger("xcube.clms")
LOG.handlers.clear() 
LOG.setLevel(logging.DEBUG)

handler = logging.StreamHandler()
handler.setLevel(logging.DEBUG)
handler.setFormatter(logging.Formatter(
    "%(asctime)s [%(levelname)s] %(name)s: %(message)s"
))
LOG.addHandler(handler)

In [ ]:
%%time
clms_data_store = new_data_store("clms", credentials=credentials)

In [ ]:
%%time
clms_data = clms_data_store.open_data("daily-surface-soil-moisture-v1.0", time_range=time_range)
clms_data

In [ ]:
# Missing dates from source - '2020-07-18', '2021-05-31' when running this for 2 years from 2020-2021

In [ ]:
clms_ssm_only = clms_data.drop_vars("ssm_noise") 
clms_ssm_only

In [ ]:
dask.config.set(scheduler="threads", num_workers=1)

In [ ]:
from xcube.core.chunk import chunk_dataset

In [ ]:
clms_ssm_only_chunked = chunk_dataset(clms_ssm_only, {"time": 2, "lat": 4144, "lon": 6832})
clms_ssm_only_chunked

In [ ]:
%%time
irr_store.write_data(clms_ssm_only_chunked, f"clms.zarr")

In [ ]:
irr_store.list_data_ids()

In [ ]:
irr_store.open_data("clms.zarr")